# Actividad 3 — Patrón Observer (Comportamiento)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Monitoreo y Gestión de Cuartos Fríos (Cadena de Frío)

---

## 1. Contexto del Problema

En un cuarto frío de vacunas o medicamentos, si la temperatura sube por encima del límite permitido (por ejemplo, más de 8°C), varios sistemas tienen que enterarse al mismo tiempo:
1. **La sirena física de la planta:** Para avisar a los operarios en sitio con sonido y luz.
2. **El bot de Telegram:** Para alertar al celular del técnico de turno.
3. **El log de auditoría:** Para guardar la traza requerida por las normas sanitarias (INVIMA).

### El problema:
Si la clase que revisa la temperatura (`MonitorCuartoFrio`) llama directamente a las funciones de la sirena, del bot de Telegram y del log dentro de su propio código:
- La clase queda sobrecargada haciendo cosas de comunicación y hardware.
- No se puede silenciar la sirena si están en mantenimiento sin modificar el código.
- Si queremos agregar un canal nuevo (ej. WhatsApp o correo), nos toca volver a editar la clase del monitor.

**Solución con Observer:**  
El monitor actúa como el **Sujeto (Publisher)** manteniendo una lista de suscriptores (`IObservadorAlarma`). Cuando la temperatura se sale de rango, el monitor solo llama a `notificar_todos()` y cada observador hace su trabajo de forma independiente.


## 2. Código Sin Patrón (Forma Incorrecta)

En esta versión el monitor tiene cableadas adentro las llamadas a cada canal. No se pueden quitar ni agregar canales en tiempo de ejecución.


In [1]:
# Código sin patrón: monitor acoplado a todos los canales

import datetime

class MonitorSinPatron:
    def __init__(self, cuarto_id: str, temp_maxima: float):
        self.cuarto_id = cuarto_id
        self.temp_maxima = temp_maxima

    # Métodos fijos dentro de la misma clase
    def _sonar_sirena(self, temp: float):
        print(f"[Sirena Planta] Alarma sonando en {self.cuarto_id} por {temp}°C")

    def _mandar_telegram(self, temp: float):
        print(f"[Telegram] Enviando mensaje a técnicos: {self.cuarto_id} subió a {temp}°C")

    def _guardar_log(self, temp: float):
        fecha = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[Auditoría] Guardando log [{fecha}] Cuarto {self.cuarto_id}: {temp}°C")

    def evaluar_temperatura(self, temp_actual: float):
        print(f"\n--- Revisando {self.cuarto_id}: {temp_actual}°C (Máx: {self.temp_maxima}°C) ---")
        if temp_actual > self.temp_maxima:
            print(">> TEMPERATURA FUERA DE RANGO <<")
            # Llamadas amarradas: si quiero apagar la sirena en mantenimiento toca editar aquí
            self._sonar_sirena(temp_actual)
            self._mandar_telegram(temp_actual)
            self._guardar_log(temp_actual)
        else:
            print("Temperatura normal.")


# Prueba sin patrón
monitor = MonitorSinPatron("Cuarto-Vacunas", temp_maxima=8.0)
monitor.evaluar_temperatura(5.0)
monitor.evaluar_temperatura(10.2)



--- Revisando Cuarto-Vacunas: 5.0°C (Máx: 8.0°C) ---
Temperatura normal.

--- Revisando Cuarto-Vacunas: 10.2°C (Máx: 8.0°C) ---
>> TEMPERATURA FUERA DE RANGO <<
[Sirena Planta] Alarma sonando en Cuarto-Vacunas por 10.2°C
[Telegram] Enviando mensaje a técnicos: Cuarto-Vacunas subió a 10.2°C
[Auditoría] Guardando log [2026-09-05 14:06:26] Cuarto Cuarto-Vacunas: 10.2°C


## 3. Código Con Patrón Observer (Forma Correcta)

Estructura de la solución:
- **`EventoAlarma` (Datos del evento):** Guarda los datos de la alerta (cuarto, temperatura, fecha, severidad).
- **`IObservadorAlarma` (Interfaz Observador):** Define el método `notificar(evento)` que todos los canales deben tener.
- **`MonitorCuartoFrio` (Sujeto):** Administra los observadores (`suscribir`, `desuscribir`) y dispara `notificar_todos()` cuando la temperatura supera el límite.
- **Observadores Concretos:** `SirenaPlanta`, `TelegramAlerta`, `AuditoriaLog` y `ValvulaAuxilio`.


In [2]:
from abc import ABC, abstractmethod
import datetime
from dataclasses import dataclass

# 1. Objeto simple con los datos del evento
@dataclass
class EventoAlarma:
    cuarto_id: str
    temp_actual: float
    temp_maxima: float
    severidad: str
    fecha_hora: str


# 2. Interfaz para los observadores
class IObservadorAlarma(ABC):
    @abstractmethod
    def notificar(self, evento: EventoAlarma):
        pass

    @abstractmethod
    def nombre_canal(self) -> str:
        pass


# 3. El Sujeto (quien monitorea y avisa a los observadores)
class MonitorCuartoFrio:
    def __init__(self, cuarto_id: str, temp_maxima: float):
        self.cuarto_id = cuarto_id
        self.temp_maxima = temp_maxima
        self._observadores: list[IObservadorAlarma] = []

    def suscribir(self, observador: IObservadorAlarma):
        if observador not in self._observadores:
            self._observadores.append(observador)
            print(f"[Monitor] Suscrito canal: {observador.nombre_canal()}")

    def desuscribir(self, observador: IObservadorAlarma):
        if observador in self._observadores:
            self._observadores.remove(observador)
            print(f"[Monitor] Desuscrito canal: {observador.nombre_canal()}")

    def notificar_todos(self, evento: EventoAlarma):
        print(f"\n>> Disparando alerta a {len(self._observadores)} observador(es)...")
        for obs in self._observadores:
            obs.notificar(evento)

    def evaluar_temperatura(self, temp_actual: float):
        print(f"\n--- Evaluando {self.cuarto_id} | Temp: {temp_actual}°C | Límite: {self.temp_maxima}°C ---")
        if temp_actual > self.temp_maxima:
            severidad = "CRÍTICA" if temp_actual - self.temp_maxima >= 3.0 else "ALTA"
            ahora = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            evento = EventoAlarma(
                cuarto_id=self.cuarto_id,
                temp_actual=temp_actual,
                temp_maxima=self.temp_maxima,
                severidad=severidad,
                fecha_hora=ahora
            )
            self.notificar_todos(evento)
        else:
            print("Estado: Temperatura dentro de lo normal.")


# 4. Observadores concretos
class SirenaPlanta(IObservadorAlarma):
    def __init__(self, nave: str):
        self.nave = nave

    def notificar(self, evento: EventoAlarma):
        print(f" [Sirena - {self.nave}] BEEP BEEP! Alarma visual y sonora activada por {evento.severidad} en {evento.cuarto_id}.")

    def nombre_canal(self) -> str:
        return f"Sirena Física ({self.nave})"


class TelegramAlerta(IObservadorAlarma):
    def __init__(self, grupo: str):
        self.grupo = grupo

    def notificar(self, evento: EventoAlarma):
        print(f" [Telegram -> {self.grupo}] Alerta: {evento.cuarto_id} está a {evento.temp_actual}°C ({evento.severidad}) a las {evento.fecha_hora}.")

    def nombre_canal(self) -> str:
        return "Telegram Bot"


class AuditoriaLog(IObservadorAlarma):
    def __init__(self, archivo: str = "auditoria_frio.log"):
        self.archivo = archivo

    def notificar(self, evento: EventoAlarma):
        print(f" [Auditoría] Registro guardado en '{self.archivo}': {evento.cuarto_id} -> {evento.temp_actual}°C [{evento.fecha_hora}]")

    def nombre_canal(self) -> str:
        return "Log de Auditoría"


class ValvulaAuxilio(IObservadorAlarma):
    """Nuevo observador para mostrar extensibilidad sin tocar el monitor"""
    def __init__(self, valvula_id: str):
        self.valvula_id = valvula_id

    def notificar(self, evento: EventoAlarma):
        if evento.severidad == "CRÍTICA":
            print(f" [Válvula Auxilio] ABRIENDO válvula {self.valvula_id} de emergencia para enfriar rápido.")

    def nombre_canal(self) -> str:
        return f"Válvula Auxilio ({self.valvula_id})"


# 5. Demostración práctica
monitor_vacunas = MonitorCuartoFrio("Cuarto-Vacunas-P1", temp_maxima=8.0)

sirena = SirenaPlanta("Nave 1")
telegram = TelegramAlerta("@TecnicosGuardia")
auditoria = AuditoriaLog()

# Suscribimos los canales iniciales
print("--- 1. Conectando canales iniciales ---")
monitor_vacunas.suscribir(sirena)
monitor_vacunas.suscribir(telegram)
monitor_vacunas.suscribir(auditoria)

# Lectura normal
monitor_vacunas.evaluar_temperatura(4.5)

# Lectura con alerta
monitor_vacunas.evaluar_temperatura(9.5)

# Entramos en mantenimiento: apagamos la sirena para no hacer ruido
print("\n--- 2. Modo Mantenimiento (apagamos sirena y conectamos válvula) ---")
monitor_vacunas.desuscribir(sirena)

# Conectamos un nuevo observador en caliente
valvula = ValvulaAuxilio("VALV-N2-01")
monitor_vacunas.suscribir(valvula)

# Desviación crítica: avisa a Telegram, Auditoría y abre válvula (sin sirena)
monitor_vacunas.evaluar_temperatura(12.0)


--- 1. Conectando canales iniciales ---
[Monitor] Suscrito canal: Sirena Física (Nave 1)
[Monitor] Suscrito canal: Telegram Bot
[Monitor] Suscrito canal: Log de Auditoría

--- Evaluando Cuarto-Vacunas-P1 | Temp: 4.5°C | Límite: 8.0°C ---
Estado: Temperatura dentro de lo normal.

--- Evaluando Cuarto-Vacunas-P1 | Temp: 9.5°C | Límite: 8.0°C ---

>> Disparando alerta a 3 observador(es)...
 [Sirena - Nave 1] BEEP BEEP! Alarma visual y sonora activada por ALTA en Cuarto-Vacunas-P1.
 [Telegram -> @TecnicosGuardia] Alerta: Cuarto-Vacunas-P1 está a 9.5°C (ALTA) a las 2026-09-05 14:06:26.
 [Auditoría] Registro guardado en 'auditoria_frio.log': Cuarto-Vacunas-P1 -> 9.5°C [2026-09-05 14:06:26]

--- 2. Modo Mantenimiento (apagamos sirena y conectamos válvula) ---
[Monitor] Desuscrito canal: Sirena Física (Nave 1)
[Monitor] Suscrito canal: Válvula Auxilio (VALV-N2-01)

--- Evaluando Cuarto-Vacunas-P1 | Temp: 12.0°C | Límite: 8.0°C ---

>> Disparando alerta a 3 observador(es)...
 [Telegram -> @Tecn

## 4. Diagrama UML

```plantuml
@startuml
skinparam classAttributeIconSize 0

class EventoAlarma {
    + cuarto_id: str
    + temp_actual: float
    + temp_maxima: float
    + severidad: str
    + fecha_hora: str
}

class MonitorCuartoFrio {
    - cuarto_id: str
    - temp_maxima: float
    - _observadores: list<IObservadorAlarma>
    + suscribir(obs: IObservadorAlarma)
    + desuscribir(obs: IObservadorAlarma)
    + notificar_todos(evento: EventoAlarma)
    + evaluar_temperatura(temp_actual: float)
}

interface IObservadorAlarma {
    + {abstract} notificar(evento: EventoAlarma)
    + {abstract} nombre_canal() : str
}

class SirenaPlanta {
    + notificar(evento: EventoAlarma)
    + nombre_canal() : str
}

class TelegramAlerta {
    + notificar(evento: EventoAlarma)
    + nombre_canal() : str
}

class AuditoriaLog {
    + notificar(evento: EventoAlarma)
    + nombre_canal() : str
}

class ValvulaAuxilio {
    + notificar(evento: EventoAlarma)
    + nombre_canal() : str
}

MonitorCuartoFrio o--> "0..*" IObservadorAlarma
IObservadorAlarma <|.. SirenaPlanta
IObservadorAlarma <|.. TelegramAlerta
IObservadorAlarma <|.. AuditoriaLog
IObservadorAlarma <|.. ValvulaAuxilio

MonitorCuartoFrio ..> EventoAlarma : crea
IObservadorAlarma ..> EventoAlarma : recibe
@enduml
```


## 5. ¿Por qué elegí Observer y no otro patrón?

Revisando los otros patrones de comportamiento:

- **¿Por qué no Strategy?**  
  Strategy se usa para intercambiar un único algoritmo a la vez (por ejemplo, elegir entre un método u otro de calcular costos). En el caso de las alarmas no queremos elegir una sola alternativa, sino que **varios canales independientes reaccionen a la vez** ante el mismo evento.

- **¿Por qué no Chain of Responsibility?**  
  Chain of Responsibility pasa la solicitud por una cadena donde normalmente uno solo de los eslabones la procesa y frena la cadena. Aquí en una emergencia térmica necesitamos que todos los canales activos actúen simultáneamente.

- **¿Por qué no Mediator?**  
  Mediator sirve para coordinar la comunicación cruzada entre muchos objetos que hablan entre sí. Aquí la comunicación va en una sola dirección: el monitor publica la alarma y los demás reaccionan.

- **Conclusión:**  
  Observer fue la mejor elección porque nos da una relación 1 a muchos desacoplada, permitiendo conectar y desconectar canales de alerta en caliente según lo que necesite la planta.
